# Aula 05 — Laboratório
## Quem paga e quem não paga

**Análise Econômica e Machine Learning** · Prof. Dr. Flavius Sodré · Economia, 8º período

Dois conjuntos de dados, o mesmo roteiro:

1. uma **carteira simulada** de crédito pessoal — ali a probabilidade verdadeira de calote é
   conhecida;
2. **30.000 cartões de crédito em Taiwan**, de abril a setembro de 2005, e o calote em
   outubro (Yeh e Lien, 2009).

Em cada um: árvores de decisão, Random Forest e Gradient Boosting; a AUC fora da amostra;
e, em Taiwan, a passagem da probabilidade prevista à decisão de conceder crédito.

**Como completar:** onde houver `# COMPLETE`, escreva o código. Onde houver `___`,
substitua. Rode as células na ordem.

## Bloco 0 — ambiente

In [ ]:
# Bloco 0 — rode esta celula e confira a saida.
import io, os, sys, zipfile, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (10, 4.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# As mesmas cores dos slides. Azul = pagou / dentro da amostra / arvore.
# Laranja = calote / fora da amostra / logit. Verde = floresta e boosting.
AZUL, LARANJA, VERDE, BORDO, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#6F1D2B", "#8A8D96"

try:
    import sklearn
    from sklearn.tree import DecisionTreeClassifier, export_text
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
    from sklearn.metrics import roc_auc_score, accuracy_score
except ImportError:
    raise ImportError("Este laboratorio precisa do scikit-learn. "
                      "Instale com:  pip install scikit-learn")


def auc(y, p):
    return float(roc_auc_score(y, p))


print("Python      :", sys.version.split()[0], "|", platform.system())
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("\nAmbiente pronto.")

## Bloco 1 — aquecimento com a verdade na mão

1.000 clientes no treino, 20.000 no teste, duas variáveis e a probabilidade verdadeira de
calote de cada um.

In [ ]:
# Bloco 1 — a carteira simulada.
#
# Duas informacoes por cliente: renda (R$ mil por mes) e comprometimento da
# renda com dividas (%). A probabilidade verdadeira de calote e conhecida:
# sobe com o comprometimento, cai com a renda e da um salto quando quem ganha
# menos de R$ 2,5 mil compromete mais de 40% da renda.

CONSTANTE = -3.0
EFEITO_COMPROMETIMENTO = 0.06
EFEITO_RENDA = 0.9
SALTO = 1.5
LIMIAR_COMPROMETIMENTO = 40.0
LIMIAR_RENDA = 2.5


def logistica(z):
    return 1 / (1 + np.exp(-z))

def prob_verdadeira(renda, comprometimento):
    """Probabilidade verdadeira de calote no mundo simulado."""
    renda = np.asarray(renda, dtype=float)
    comprometimento = np.asarray(comprometimento, dtype=float)
    acima = logistica((comprometimento - LIMIAR_COMPROMETIMENTO) / 2.0)
    pobre = logistica((LIMIAR_RENDA - renda) / 0.3)
    z = (CONSTANTE + EFEITO_COMPROMETIMENTO * (comprometimento - 30)
         - EFEITO_RENDA * np.log(renda / 3) + SALTO * acima * pobre)
    return logistica(z)

def gerar_carteira(n, semente):
    """`n` clientes de crédito pessoal: renda (R$ mil por mês), comprometimento
    da renda com dívidas (%) e se deu calote (1) ou não (0)."""
    rng = np.random.default_rng(semente)
    renda = np.clip(np.exp(rng.normal(np.log(3.2), 0.55, n)), 0.8, 20)
    comprometimento = np.clip(rng.beta(2.0, 3.0, n) * 80, 1, 78)
    p = prob_verdadeira(renda, comprometimento)
    calote = (rng.random(n) < p).astype(int)
    X = pd.DataFrame({"renda": renda.round(2), "comprometimento": comprometimento.round(1)})
    return X, calote


X_sim, y_sim = gerar_carteira(1000, 100)                 # 1.000 clientes no treino
X_sim_teste, y_sim_teste = gerar_carteira(20000, 999)    # 20.000 que o modelo nao vera
p_sim_teste = prob_verdadeira(X_sim_teste["renda"], X_sim_teste["comprometimento"])
TETO = auc(y_sim_teste, p_sim_teste)

print(f"treino: {len(y_sim)} clientes, {100 * y_sim.mean():.1f}% de calote")
print(f"teste : {len(y_sim_teste)} clientes")
print(f"teto  : AUC {TETO:.3f} (quem conhece a probabilidade verdadeira)")
X_sim.head()

In [ ]:
# Bloco 1 — uma arvore de duas perguntas. COMPLETE.
#
# Dois niveis, folhas de pelo menos 30 clientes. Depois, leia as regras.

# COMPLETE
arvore_2 = DecisionTreeClassifier(max_depth=___, min_samples_leaf=___, random_state=0)
arvore_2.fit(___, ___)

# COMPLETE: imprima as regras com export_text (show_weights=True mostra
# quantos pagaram e quantos deram calote em cada folha)
print(___)

In [ ]:
# Bloco 1 — a AUC pela definicao. COMPLETE.
#
# A AUC e a chance de o modelo dar risco maior a quem deu calote do que a quem
# pagou, num par sorteado. Confira com 2.000 clientes de teste.
p2 = arvore_2.predict_proba(X_sim_teste.iloc[:2000])[:, 1]
y2 = y_sim_teste[:2000]
maus, bons = p2[y2 == 1], p2[y2 == 0]

# COMPLETE: fracao dos pares (mau, bom) em que o mau tem risco maior;
# empate conta meio
auc_pares = ___

print(f"AUC pela contagem de pares: {auc_pares:.4f}")
print(f"AUC do scikit-learn       : {auc(y2, p2):.4f}")

In [ ]:
# Bloco 1 — o botao de profundidade. COMPLETE.
#
# Para cada profundidade de 1 a 15 (folhas sem minimo), ajuste a arvore e
# meca a AUC dentro da amostra (X_sim) e fora dela (X_sim_teste).
profundidades = range(1, 16)
auc_dentro, auc_fora = [], []
for d in profundidades:
    # COMPLETE
    t = ___
    auc_dentro.append(___)
    auc_fora.append(___)

fig, ax = plt.subplots()
ax.plot(profundidades, auc_dentro, "o-", color=AZUL, lw=2.5, label="dentro da amostra")
ax.plot(profundidades, auc_fora, "o-", color=LARANJA, lw=2.5, label="fora da amostra")
ax.axhline(TETO, color=CINZA, ls="--", label="teto (probabilidade verdadeira)")
ax.set_xlabel("profundidade maxima"); ax.set_ylabel("AUC"); ax.legend(frameon=False)
plt.show()
print("profundidade de maior AUC fora da amostra:", list(profundidades)[int(np.argmax(auc_fora))])

In [ ]:
# Bloco 1 — junte as arvores. COMPLETE.
#
# Floresta: 500 arvores, folhas de pelo menos 20 clientes.
# Boosting: 100 rodadas, taxa de aprendizado 0,05, arvores de 2 niveis,
# folhas de pelo menos 20 clientes.

# COMPLETE
floresta_sim = RandomForestClassifier(n_estimators=___, min_samples_leaf=___,
                                      random_state=0, n_jobs=-1).fit(X_sim, y_sim)
boosting_sim = GradientBoostingClassifier(n_estimators=___, learning_rate=___, max_depth=___,
                                          min_samples_leaf=20, random_state=0).fit(X_sim, y_sim)

# COMPLETE: AUC fora da amostra de cada um, e quanto falta para o teto
___

## Bloco 2 — Taiwan: da planilha à matriz do modelo

Cada linha é um cliente de cartão de crédito de um banco de Taiwan. As colunas `PAY_0` e
`PAY_2` a `PAY_6` são a situação do cartão de setembro a abril de 2005: −2, −1 e 0 indicam
conta em dia; 1 a 8, meses de atraso. `BILL_AMT` são as faturas e `PAY_AMT`, os pagamentos,
em novos dólares de Taiwan (NT$). O alvo é o calote em outubro.

In [ ]:
# Bloco 2 — Plano B. Rode e siga em frente; nao precisa ler.
#
# Resumo da base real (frequencias de limite, idade, sexo, escolaridade e
# estado civil; como a situacao do cartao passa de um mes para o outro; uso do
# limite e fracao da fatura paga) e um logit simples do calote. Serve so para
# gerar clientes inventados com a mesma estrutura.

ALVO = 'default.payment.next.month'
ATRASOS = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
FATURAS = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
PAGAMENTOS = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
COLUNAS = ['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default.payment.next.month']
ESTADOS = [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]

ESP_TAIWAN = {'LIMIT_BAL': [[10000, 16000, 20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000,
                110000, 120000, 130000, 140000, 150000, 160000, 170000, 180000, 190000, 200000,
                210000, 220000, 230000, 240000, 250000, 260000, 270000, 280000, 290000, 300000,
                310000, 320000, 327680, 330000, 340000, 350000, 360000, 370000, 380000, 390000,
                400000, 410000, 420000, 430000, 440000, 450000, 460000, 470000, 480000, 490000,
                500000, 510000, 520000, 530000, 540000, 550000, 560000, 570000, 580000, 590000,
                600000, 610000, 620000, 630000, 640000, 650000, 660000, 670000, 680000, 690000,
                700000, 710000, 720000, 730000, 740000, 750000, 760000, 780000, 800000, 1000000],
               [0.01643, 7e-05, 0.06587, 0.05367, 0.00767, 0.11217, 0.0275, 0.02437, 0.05223,
                0.0217, 0.03493, 0.0196, 0.0242, 0.0243, 0.02497, 0.037, 0.02313, 0.01773, 0.03317,
                0.00763, 0.05093, 0.02433, 0.01563, 0.02457, 0.02063, 0.01167, 0.01737, 0.00793,
                0.01643, 0.0116, 0.01847, 0.00907, 0.0104, 3e-05, 0.00577, 0.00723, 0.0077, 0.02937,
                0.00237, 0.0052, 0.0058, 0.00903, 0.0026, 0.0056, 0.00277, 0.00277, 0.00537,
                0.00267, 0.00267, 0.00263, 0.00213, 0.02407, 0.00063, 0.00067, 0.00033, 0.0002,
                0.0007, 0.00033, 0.00027, 0.00037, 0.0002, 0.00053, 0.00037, 0.0003, 0.00023,
                0.00023, 0.0001, 0.0001, 0.0001, 0.00013, 3e-05, 0.00027, 0.0002, 0.0001, 7e-05,
                7e-05, 0.00013, 3e-05, 7e-05, 7e-05, 3e-05]],
 'AGE': [[21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42,
          43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64,
          65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 79],
         [0.00223, 0.01867, 0.03103, 0.03757, 0.03953, 0.04187, 0.04923, 0.04697, 0.0535, 0.0465,
          0.04057, 0.0386, 0.0382, 0.03873, 0.0371, 0.03693, 0.0347, 0.03147, 0.0318, 0.029,
          0.02747, 0.02647, 0.02233, 0.02333, 0.02057, 0.019, 0.0167, 0.01553, 0.01507, 0.0137,
          0.01133, 0.01013, 0.01083, 0.00823, 0.00697, 0.00593, 0.00407, 0.00407, 0.00277, 0.00223,
          0.00187, 0.00147, 0.00103, 0.00103, 0.0008, 0.00083, 0.00053, 0.00017, 0.0005, 0.00033,
          0.0001, 0.0001, 0.00013, 3e-05, 0.0001, 3e-05]],
 'SEX': [[1, 2], [0.39627, 0.60373]],
 'EDUCATION': [[0, 1, 2, 3, 4, 5, 6], [0.00047, 0.35283, 0.46767, 0.1639, 0.0041, 0.00933, 0.0017]],
 'MARRIAGE': [[0, 1, 2, 3], [0.0018, 0.4553, 0.53213, 0.01077]],
 'inicio': [0.16317, 0.19133, 0.54287, 0.0, 0.0922, 0.00613, 0.00163, 0.00043, 0.00063, 0.00153,
            7e-05],
 'transicao': [[0.8128, 0.0682, 0.05038, 0.05694, 0.01168, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
               [0.06576, 0.75689, 0.12005, 0.02145, 0.03585, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
               [5e-05, 0.05058, 0.88872, 7e-05, 0.06058, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
               [0.00029, 0.00029, 0.00029, 0.99707, 0.00029, 0.00029, 0.00029, 0.00029, 0.00029,
                0.00029, 0.00029],
               [0.00061, 0.08014, 0.17267, 0.10284, 0.58047, 0.06326, 0.0, 0.0, 0.0, 0.0, 0.0],
               [1e-05, 0.07581, 0.08303, 0.09837, 0.32669, 0.15884, 0.2572, 1e-05, 1e-05, 1e-05,
                1e-05],
               [3e-05, 0.02124, 0.02124, 0.08488, 0.22542, 0.07693, 0.28111, 0.28907, 3e-05, 3e-05,
                3e-05],
               [9e-05, 0.02709, 0.02709, 0.06309, 0.16209, 0.06309, 0.09909, 0.10809, 0.45009,
                9e-05, 9e-05],
               [0.00016, 0.016, 0.016, 0.03185, 0.07939, 0.016, 0.016, 0.04769, 0.06354, 0.7132,
                0.00016],
               [5e-05, 5e-05, 5e-05, 0.00483, 0.26785, 0.00961, 5e-05, 5e-05, 0.00483, 0.6026,
                0.11004],
               [0.0011, 0.0011, 0.0011, 0.11087, 0.33041, 0.0011, 0.11087, 0.0011, 0.0011, 0.11087,
                0.33041]],
 'uso': {'-2': [0.0, 0.0933],
         '-1': [0.0136, 0.1606],
         '0': [0.5141, 0.3382],
         '1': [0.0313, 0.4004],
         '2': [0.6213, 0.3567],
         '3': [0.7173, 0.3796],
         '4': [0.8591, 0.3797],
         '5': [0.9343, 0.3888],
         '6': [0.9898, 0.4309],
         '7': [0.0808, 0.374],
         '8': [1.0963, 0.3876]},
 'pago': {'-2': 1.0,
          '-1': 1.0,
          '0': 0.0491,
          '1': 0.0375,
          '2': 0.0433,
          '3': 0.0,
          '4': 0.0,
          '5': 0.0,
          '6': 0.0,
          '7': 0.0,
          '8': 0.0},
 'calote': [1.59163, -0.14007, -0.37803, 0.57321, 1.95988, 1.97565, 1.37242, 0.5472, -0.29859,
            0.00842]}


def logistica(z):
    return 1 / (1 + np.exp(-z))

def taiwan_sintetica(ESP, n=30000, semente=2005):
    """Plano B: clientes INVENTADOS com as mesmas colunas, os mesmos tipos e
    os mesmos códigos da base real. A situação de cada mês segue a do mês
    anterior; a fatura acompanha o limite e a situação; o calote sai de um
    logit da situação em setembro e em agosto, do limite e da idade."""
    rng = np.random.default_rng(semente)

    def sorteio(chave, k):
        v, p = ESP[chave]
        p = np.array(p, dtype=float)
        return rng.choice(np.array(v), k, p=p / p.sum())

    d = {"ID": np.arange(1, n + 1)}
    for c in ("LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"):
        d[c] = sorteio(c, n)
    ini = np.array(ESP["inicio"], dtype=float)
    T = np.array(ESP["transicao"], dtype=float)
    sit = np.zeros((n, 6), dtype=int)                       # abr → set
    sit[:, 0] = rng.choice(ESTADOS, n, p=ini / ini.sum())
    for t in range(1, 6):
        u = rng.random(n)
        acum = np.cumsum(T[sit[:, t - 1] + 2], axis=1)
        sit[:, t] = np.array(ESTADOS)[(u[:, None] > acum).sum(axis=1).clip(0, 10)]
    for j, c in enumerate(ATRASOS):                          # PAY_0 = setembro
        d[c] = sit[:, 5 - j]
    lim = d["LIMIT_BAL"].astype(float)
    for j, c in enumerate(FATURAS):
        e = sit[:, 5 - j]
        med = np.array([ESP["uso"][str(v)][0] for v in e])
        dp = np.array([ESP["uso"][str(v)][1] for v in e])
        uso = np.clip(med + dp * rng.normal(0, 1, n), -0.05, 1.3)
        uso = np.where(e == -2, np.where(rng.random(n) < 0.7, 0, uso * 0.1), uso)
        d[c] = np.round(lim * uso).astype(int)
    for j, c in enumerate(PAGAMENTOS):
        if j < 5:
            e = sit[:, 5 - j]
            fr = np.array([ESP["pago"][str(v)] for v in e])
            base = np.maximum(d[FATURAS[j + 1]], 0)
            d[c] = np.round(base * fr * np.exp(rng.normal(0, 0.5, n))).astype(int)
        else:
            d[c] = np.round(np.maximum(d[FATURAS[5]], 0) * 0.1).astype(int)
    b = np.array(ESP["calote"])
    A = np.column_stack([np.ones(n)]
                        + [(np.clip(d["PAY_0"], -2, 4) == e).astype(float) for e in range(-2, 5) if e != -1]
                        + [(d["PAY_2"] >= 2).astype(float), np.log(lim), d["AGE"]])
    d[ALVO] = (rng.random(n) < logistica(A @ b)).astype(int)
    return pd.DataFrame(d)[COLUNAS]


print("Plano B pronto (so e usado se a base real nao carregar).")

In [ ]:
# Bloco 2 — os cartoes de credito de Taiwan (Yeh e Lien, 2009).
#
# Tres caminhos, nesta ordem:
#   1. o arquivo credito_taiwan.csv, na pasta Dados do repositorio;
#   2. a planilha original da UCI (zip com .xls; precisa do pacote xlrd);
#   3. o pacote `ucimlrepo`, se estiver instalado.
# Se os tres falharem, segue o PLANO B (celula acima), com dados inventados.

ARQUIVO = "../Dados/credito_taiwan.csv"
URL = "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip"


def padronizar(d):
    """Mesmas colunas, na mesma ordem, venha a base de onde vier."""
    d = d.rename(columns={"default payment next month": ALVO})
    if list(d.columns) != COLUNAS:
        if len(d.columns) == len(COLUNAS) - 1:          # veio sem a coluna de ID
            d.insert(0, "ID", np.arange(1, len(d) + 1))
        if len(d.columns) == len(COLUNAS):              # X1..X23, Y -> nomes da UCI
            d.columns = COLUNAS
    if list(d.columns) != COLUNAS or len(d) < 1000:
        raise ValueError("a tabela nao tem a estrutura esperada")
    return d.astype(int)


def ler_planilha_uci(conteudo):
    with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
        nome = [n for n in z.namelist() if n.endswith(".xls")][0]
        return pd.read_excel(io.BytesIO(z.read(nome)), header=1)


def carregar_taiwan(arquivo=ARQUIVO, url=URL, timeout=120):
    if os.path.exists(arquivo):
        return padronizar(pd.read_csv(arquivo)), "arquivo local " + arquivo
    try:
        import requests
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        return padronizar(ler_planilha_uci(r.content)), "UCI (download)"
    except Exception as erro:
        print("download falhou:", type(erro).__name__, str(erro)[:80])
    try:
        from ucimlrepo import fetch_ucirepo
        return padronizar(fetch_ucirepo(id=350).data.original), "pacote ucimlrepo"
    except Exception:
        pass
    print("=" * 70)
    print("PLANO B: os dados abaixo sao INVENTADOS. Mesmas colunas, mesmos")
    print("tipos e mesmos codigos da base real; clientes e relacoes sao")
    print("sinteticos. Os numeros NAO valem como resultado sobre Taiwan.")
    print("=" * 70)
    return taiwan_sintetica(ESP_TAIWAN), "PLANO B (sintetico)"


cartoes, FONTE = carregar_taiwan()
print(f"fonte: {FONTE}")
print(f"{cartoes.shape[0]} clientes, {cartoes.shape[1]} colunas")

In [ ]:
# Bloco 2 — o primeiro olhar. COMPLETE.
#
# Responda com codigo: qual a taxa de calote? E a taxa de calote por situacao
# do cartao em setembro (PAY_0, juntando 4 meses ou mais num grupo so)?

# COMPLETE
___

In [ ]:
# Bloco 2 — da tabela crua a matriz do modelo. COMPLETE.
def preparar_taiwan(dados):
    d = dados.copy()
    y = d[ALVO].to_numpy().astype(int)
    esc = d["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
    # COMPLETE: limite, idade, atrasos, faturas e pagamentos — SEM sexo e estado civil
    X = d[___].astype(float)
    X["EDU_universidade"] = (esc == 2).astype(float)
    X["EDU_medio"] = (esc == 3).astype(float)
    X["EDU_outros"] = (esc == 4).astype(float)
    treino = np.zeros(len(y), dtype=bool)
    # COMPLETE: 75% para treino, sorteio estratificado pelo alvo, random_state=2026
    idx_tr, _ = train_test_split(np.arange(len(y)), test_size=___, stratify=___,
                                 random_state=2026)
    treino[idx_tr] = True
    return X, y, treino


X, y, treino = preparar_taiwan(cartoes)
print(f"{X.shape[0]} clientes, {X.shape[1]} colunas no modelo")
print(f"treino: {treino.sum()} ({100 * y[treino].mean():.1f}% de calote) | "
      f"teste: {(~treino).sum()} ({100 * y[~treino].mean():.1f}% de calote)")

## Bloco 3 — cinco modelos e uma linha de base

In [ ]:
# Bloco 3 — as ferramentas. Leia, nao precisa alterar.

def faixas_de_atraso(X):
    """Cada coluna de atraso vira três colunas 0/1 — um mês, dois meses, três
    ou mais —, com "em dia" como referência. É o degrau escrito à mão, para o
    logit poder vê-lo."""
    Z = X.copy()
    for c in ATRASOS:
        v = Z.pop(c)
        Z[c + "_1m"] = (v == 1).astype(float)
        Z[c + "_2m"] = (v == 2).astype(float)
        Z[c + "_3m+"] = (v >= 3).astype(float)
    return Z

def logit(X, y):
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(X, y)

def floresta(n_arvores=500, folha=20, semente=0):
    """Random Forest: cada árvore cresce numa amostra sorteada com reposição e,
    em cada corte, escolhe entre um sorteio das variáveis. Folhas de pelo
    menos `folha` clientes."""
    return RandomForestClassifier(n_estimators=n_arvores, min_samples_leaf=folha,
                                  max_features="sqrt", random_state=semente,
                                  n_jobs=-1)

def boosting_parada(semente=0):
    """Boosting com parada antecipada: separa 20% do treino e para quando 50
    rodadas seguidas não melhoram o ajuste nessa parte."""
    return GradientBoostingClassifier(n_estimators=1000, learning_rate=0.05, max_depth=3,
                                      subsample=0.8, validation_fraction=0.2,
                                      n_iter_no_change=50, random_state=semente)


print("Ferramentas prontas.")

In [ ]:
# Bloco 3 — a linha de base. COMPLETE.
Xtr, ytr, Xte, yte = X[treino], y[treino], X[~treino], y[~treino]

# COMPLETE: o "modelo" que preve que ninguem da calote. Qual a acuracia no
# teste? E a AUC? (dica: uma probabilidade igual para todos)
ninguem = np.zeros(len(yte))
print(f"acuracia de 'ninguem da calote': {___:.3f}")
print(f"AUC de 'ninguem da calote'     : {___:.3f}")

In [ ]:
# Bloco 3 — os cinco modelos. COMPLETE.
#
# A arvore tem a profundidade e o tamanho minimo de folha escolhidos por
# validacao cruzada de cinco dobras. Leva cerca de um minuto.
busca = GridSearchCV(DecisionTreeClassifier(random_state=0),
                     {"max_depth": list(range(2, 13)), "min_samples_leaf": [1, 20, 50, 100]},
                     scoring="roc_auc", cv=StratifiedKFold(5, shuffle=True, random_state=0))

# COMPLETE: ajuste os cinco no treino
modelos = {
    "logit": ___,
    "logit com faixas": ___,
    "arvore": ___,
    "floresta": ___,
    "boosting": ___,
}

# COMPLETE: probabilidade prevista no teste (cuidado: o logit com faixas
# precisa de faixas_de_atraso(Xte))
prob = ___

tabela = pd.DataFrame({
    "AUC": {k: auc(yte, p) for k, p in prob.items()},
    "acuracia": {k: accuracy_score(yte, p >= 0.5) for k, p in prob.items()},
}).round(3)
print(tabela)

### ✔ CHECKPOINT I

Com os códigos de atraso como estão, o logit faz a menor AUC da tabela. Com uma coluna para
cada faixa de atraso, quase alcança a árvore.

Em duas frases: por que a árvore não precisou das faixas? O que isso diz sobre quando vale
trocar o logit por um modelo de árvores?

## Bloco 4 — da probabilidade à decisão

O banco aprova quem tem probabilidade prevista de calote até um limiar. Cada bom pagador
aprovado rende **1 margem**; cada calote aprovado custa **4 margens**.

In [ ]:
# Bloco 4 — o lucro de cada limiar. COMPLETE.
def lucro_por_limiar(p, y, margem=1.0, perda=4.0, limiares=np.round(np.linspace(0, 1, 101), 2)):
    """Lucro da carteira aprovada, em margens por 1.000 clientes avaliados."""
    saida = []
    for t in limiares:
        aprovado = p <= t
        # COMPLETE: margem vezes os bons aprovados, menos perda vezes os maus aprovados
        saida.append(___)
    return limiares, np.array(saida)


limiares, lucro = lucro_por_limiar(prob["boosting"], yte)
# COMPLETE: o limiar de maior lucro, o lucro com corte em 0,5 e o de aprovar todos
___

In [ ]:
# Bloco 4 — mude a perda. COMPLETE.
#
# Para perdas de 2, 4 e 8 margens: o limiar otimo, o limiar teorico, a
# fracao de clientes aprovados e a taxa de calote entre os aprovados.
p = prob["boosting"]
linhas = []
for perda in (2, 4, 8):
    lim, luc = lucro_por_limiar(p, yte, perda=perda)
    t = lim[int(np.argmax(luc))]
    # COMPLETE
    linhas.append({"perda": perda, "limiar otimo": t, "teorico": ___,
                   "aprovados": ___, "calote entre aprovados": ___})
print(pd.DataFrame(linhas).round(3))

## Bloco 5 — ler o modelo e os dados por trás dele

In [ ]:
# Bloco 5 — as duas perguntas. COMPLETE.
#
# Arvore de dois niveis, folhas de pelo menos 300 clientes, no treino de
# Taiwan. Imprima as regras.

# COMPLETE
arvore_tw = ___
print(export_text(arvore_tw, feature_names=list(X.columns), show_weights=True))

In [ ]:
# Bloco 5 — o limite de credito. COMPLETE.
#
# Taxa de calote por faixa de limite (NT$): ate 50 mil, 50 a 100 mil,
# 100 a 200 mil, 200 a 300 mil, acima de 300 mil.
cortes = [0, 50000, 100000, 200000, 300000, 10 ** 7]

# COMPLETE
___

### Responda

Escreva as regras da árvore de duas perguntas como se fossem um manual de crédito. Depois,
em uma frase: quem decidiu o limite de crédito de cada cliente — e o que isso muda na
leitura da tabela acima?

### ✔ CHECKPOINT II

O banco quer usar a floresta para decidir quem recebe cartão **pela primeira vez**.

Em duas frases: o que impede? Use o que o modelo usa para prever e quem está na base de
treino.

### Responda

> A Receita Federal quer usar o mesmo roteiro para escolher **quais contribuintes fiscalizar
> primeiro**, prevendo a chance de inadimplência tributária. O que fariam o papel da
> "margem" e da "perda" do Bloco 4? E que cuidado a base de treino exigiria?